# Step 4. 나만의 MCP 서버: Memory MCP
Step 2의 "저장하고 찾기"를 **MCP(Model Context Protocol) 서버**로 만듭니다. 한 번 만들어 두면 Claude, ChatGPT, 로컬 모델 어디에나 붙일 수 있습니다.

## 핵심 개념
- **MCP 서버**: AI가 쓸 수 있는 도구(Tool)를 제공하는 프로그램
- **MCP 클라이언트**: 도구 목록을 받아 와서 호출하는 쪽 (Claude Desktop, Claude Code, 이 노트북)
- **도구 설명이 곧 사용 설명서**: 함수 위의 설명글(docstring)을 AI가 읽고 언제 쓸지 판단합니다.

이 노트북에서는 AI 없이, 우리가 직접 클라이언트가 되어 도구를 불러 봅니다.

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1분쯤 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "mcp>=2.3", "chromadb", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 서버 코드 읽기
`memory_mcp.py`는 80줄 남짓입니다. `@server.tool()`이 붙은 함수 네 개가 도구입니다.

In [ ]:
import os

SERVER_PATH = os.path.abspath("../memory_mcp.py")
if not os.path.exists(SERVER_PATH):
    raise SystemExit(f"❌ MCP 서버 파일을 찾을 수 없습니다: {SERVER_PATH}")
print(f"✅ MCP 서버 파일 확인: {SERVER_PATH}\n")

code = open(SERVER_PATH, encoding="utf-8").read()
start = code.index("@server.tool()")
print(code[start:start + 900])   # 첫 번째 도구 remember와 두 번째 도구 recall 부분

## 2. 서버에 연결해서 도구 목록 받기
`Client`가 서버 프로그램을 실행하고 대화를 시작합니다. AI도 똑같이 이 목록을 받아서 무엇을 쓸지 고릅니다.

In [ ]:
import sys
from mcp import Client, StdioServerParameters

server = StdioServerParameters(command=sys.executable, args=[SERVER_PATH], env=os.environ.copy())

async with Client(server) as mcp_client:
    result = await mcp_client.list_tools()
    print(f"✅ 도구 {len(result.tools)}개를 받았습니다.\n")
    for t in result.tools:
        args = ", ".join(t.input_schema.get("properties", {}).keys())
        print(f"🔧 {t.name}({args})")
        print(f"   {t.description}\n")

## 3. 도구 직접 불러 보기
AI가 할 일을 우리가 손으로 해 봅니다. 저장한 기억은 `~/.lab-mcp/memory` 폴더에 남습니다.

In [ ]:
async def call(name, **args):
    """MCP 도구를 하나 호출하고 결과를 돌려준다."""
    async with Client(server) as mcp_client:
        r = await mcp_client.call_tool(name, args)
        return r.structured_content["result"] if r.structured_content else r.content[0].text


# TODO: 기억할 내용을 바꿔 보세요!
print(await call("remember", content="다음 주 화요일까지 RAG 과제 제출", tags="수업"))
print(await call("remember", content="병원 주차장은 지하 2층", tags="생활"))

In [ ]:
# TODO: 질문을 바꿔 보세요!
for item in await call("recall", query="숙제 마감이 언제지?", k=2):
    print(f"거리 {item['distance']}  [{item['tags']}]  {item['content']}")

In [ ]:
print("최근 30일의 기억:")
for item in await call("list_memories", days=30):
    print(f"  {item['created']}  {item['id']}  {item['content']}")

## 정리
- MCP 서버는 "도구 이름 + 설명 + 인자"를 알려 주고, 호출을 받아 실행합니다.
- 다음 노트북에서는 **LLM이 이 목록을 보고 스스로** 도구를 고릅니다. → Step 5